# DeFT: Medical Image Denoising Demo

![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)

**Descriptor-Forked Test-Time Adaptation** — source-free, single-image denoising adaptation.

1. Set runtime to GPU: `Runtime` → `Change runtime type` → `T4 GPU`
2. Run all cells below
3. Upload your noisy medical image and see the denoised result


In [ ]:
# Clone repository and install dependencies
!git clone https://github.com/<user>/<repo>.git
%cd <repo>
!pip install -r requirements-minimal.txt
!pip install gdown


In [ ]:
# Download pretrained backbone checkpoint
!mkdir -p checkpoints
!gdown --id 372932474 -O checkpoints/unet_source_checkpoint.pt

import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    !nvidia-smi --query-gpu=memory.used --format=csv,noheader


In [ ]:
# Load DeFT model
from deft import DeFT, DeFTBackbone
import numpy as np

backbone = DeFTBackbone(in_channels=1, out_channels=64)
model = DeFT(denoiser=backbone)
model.cuda()
print("Model loaded. Ready for adaptation.")


In [ ]:
# Upload your noisy medical image and denoise it
from google.colab import files
import matplotlib.pyplot as plt

uploaded = files.upload()

for fn in uploaded.keys():
    # Load image
    img = np.load(fn) if fn.endswith(".npy") else plt.imread(fn)
    if img.ndim == 2:
        img = img[np.newaxis, ...]
    img = img.astype(np.float32)
    
    # Normalize if needed
    if img.max() > 1.0:
        img = img / 255.0
    
    noisy = torch.from_numpy(img).float().cuda()
    noisy = noisy.unsqueeze(0) if noisy.ndim == 3 else noisy
    
    print(f"Adapting {fn} (shape {list(noisy.shape)}) ...")
    with torch.no_grad():
        denoised = model.adapt(noisy, steps=5)
    
    # Display results
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    ax1.imshow(noisy.squeeze().cpu().numpy(), cmap="gray")
    ax1.set_title("Noisy Input")
    ax1.axis("off")
    ax2.imshow(denoised.squeeze().cpu().numpy(), cmap="gray")
    ax2.set_title("DeFT Denoised")
    ax2.axis("off")
    plt.show()
    
    # Save result
    out = denoised.squeeze().cpu().numpy()
    np.save(f"denoised_{fn}", out)
    print(f"Saved to denoised_{fn}")
